In [ ]:
from pathlib import Path
import os
from os import PathLike
from typing import Optional, Sequence, Union
from fastcore.basics import patch

from trouver.obsidian.vault import VaultNote, note_name_from_path
from trouver.obsidian.links import ObsidianLink, LinkType, replace_links_in_text

In [ ]:
import shutil
import tempfile
from unittest import mock

from fastcore.test import *
from nbdev.showdoc import show_doc

from trouver.helper.tests import _test_directory

## Creating/deleting/moving the note

If a `VaultNote` object represents a non-existent file, then the file can be created with empty content. The cache is updated with this single new entry, but the rest of the cache remains the same.

If the file exists, then a `FileExistsError` is raised.

If the specified directory for the file does not exist, then a `FileNotFoundError` is raised.

In [ ]:
#| export obsidian.vault
@patch
def create(self: VaultNote):
    # TODO: consider using _check_name_exists_and_unique_in_vault_cache method.
    r"""Create the note if it does not exist.
    
    The directory of the file needs to be created separately
    beforehand.

    If the file exists, then a FileExistsError is raised and
    the file modification time is not changed.
    
    **Raises**

    - FileExistsError
        - If the file already exists.
    - FileNotFoundError
        - If the directory of the file does not already exist.
    """
    Path(self.path()).touch(exist_ok=False)
    self.__class__._add_single_entry_to_cache(
        self.vault, self.rel_path)

In [ ]:
show_doc(VaultNote.create)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L610){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.create

>      VaultNote.create ()

*Create the note if it does not exist.

The directory of the file needs to be created separately
beforehand.

If the file exists, then a FileExistsError is raised and
the file modification time is not changed.

**Raises**

- FileExistsError
    - If the file already exists.
- FileNotFoundError
    - If the directory of the file does not already exist.*

In [ ]:
#| export obsidian.vault
@patch
def delete(self: VaultNote):
    r"""Delete the note if it exists.
    
    This updates the cache if necessary
    """
    if self.exists(update_cache=True):
        os.remove(self.path())
        self.__class__._remove_single_entry_from_cache(self.vault, self.rel_path)

In [ ]:
show_doc(VaultNote.delete)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L633){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.delete

>      VaultNote.delete ()

*Delete the note if it exists.

This updates the cache if necessary*

In [ ]:
#| export obsidian.vault
@patch
def move_to(self: VaultNote,
            rel_path: PathLike # The path in which to rename the path to `self` as, relative to `self.vault`.
            ) -> None:
    r"""Move/rename the note to the specified location in the vault,
    assuming that it exists.

    Nothing is done if the note does not exist.
    """
    if not self.exists():
        return
    os.rename(self.path(), Path(self.vault) / rel_path)
    self.__class__._remove_single_entry_from_cache(
        self.vault, Path(self.rel_path))
    self.__class__._add_single_entry_to_cache(
        self.vault, Path(rel_path))
    self.rel_path = str(rel_path)
    self.name = note_name_from_path(self.rel_path)
    
@patch
def move_to_folder(self: VaultNote,
                    rel_dir: PathLike # The path of the directory in which to move `self` to, relative to `self.vault`.
                    ) -> None:
    # TODO: consider using _check_name_exists_and_unique_in_vault_cache method.
    r"""Move the note to the specified folder in the vault, assuming that
    it exists.
    """
    self.move_to(Path(rel_dir) / f'{self.name}.md')

In [ ]:
show_doc(VaultNote.move_to)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L644){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.move_to

>      VaultNote.move_to (rel_path:os.PathLike)

*Move/rename the note to the specified location in the vault,
assuming that it exists.

Nothing is done if the note does not exist.*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| rel_path | PathLike | The path in which to rename the path to `self` as, relative to `self.vault`. |
| **Returns** | **None** |  |

In [ ]:
show_doc(VaultNote.move_to_folder)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L663){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.move_to_folder

>      VaultNote.move_to_folder (rel_dir:os.PathLike)

*Move the note to the specified folder in the vault, assuming that
it exists.*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| rel_dir | PathLike | The path of the directory in which to move `self` to, relative to `self.vault`. |
| **Returns** | **None** |  |

In [ ]:
#| export obsidian.vault
@patch
def rename(
        self: VaultNote,
        new_name: str,  # The new name to give the note('s file). Should not include the `.md` extension.`
        replace_links_in_vault: bool = True  # If `True`, then replace the links in the vault pointing to `note` to reflect the new name.
        ):

    # TODO: consider using _check_name_exists_and_unique_in_vault_cache method.
    r"""
    Rename the file underlying `self` to `new_name`. The directory that the file is in remains unchanged.

    Assumes that
    
    1. the name of `self` is unique among note names in `self.vault`s
    2. No pre-existing note in `vault` has `new_name` as its name. Use the `note_name_unique`
        function to check whether or not this is the case.
    3. the class' `.cache` accurately reflects the files in `vault`

    """
    parent_dir = Path(os.path.dirname(self.rel_path))
    old_wikilink_pattern_by_name = ObsidianLink(
        is_embedded=-1, file_name=self.name, anchor=-1, custom_text=-1)
    old_markdownlink_pattern_by_name_1 = ObsidianLink(
        is_embedded=-1, file_name=f'{self.name}.md', anchor=-1, custom_text=-1, link_type=LinkType.MARKDOWN)
    old_markdownlink_pattern_by_name_2 = ObsidianLink(
        is_embedded=-1, file_name=f'{self.name}', anchor=-1, custom_text=-1, link_type=LinkType.MARKDOWN)
    self.move_to(parent_dir / f'{new_name}.md')
    if not replace_links_in_vault:
        return
    for name, paths in self.__class__.cache[str(self.vault)].items():
        for path in paths:
            other_note = VaultNote(vault=self.vault, rel_path=path)
            if not other_note.exists():
                continue
            original_text = other_note.text()
            text = original_text
            text = replace_links_in_text(text, old_wikilink_pattern_by_name, new_link_name=new_name)
            text = replace_links_in_text(text, old_markdownlink_pattern_by_name_1, new_link_name=f'{new_name}.md')
            text = replace_links_in_text(text, old_markdownlink_pattern_by_name_2, new_link_name=f'{new_name}')
            if text != original_text:
                with open(other_note.path(), 'w', encoding='utf-8') as file:
                    file.write(text)
                    file.close()

In [ ]:
show_doc(VaultNote.rename)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L674){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.rename

>      VaultNote.rename (new_name:str, replace_links_in_vault:bool=True)

*Rename the file underlying `self` to `new_name`. The directory that the file is in remains unchanged.

Assumes that

1. the name of `self` is unique among note names in `self.vault`s
2. No pre-existing note in `vault` has `new_name` as its name. Use the `note_name_unique`
    function to check whether or not this is the case.
3. the class' `.cache` accurately reflects the files in `vault`*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| new_name | str |  | The new name to give the note('s file). Should not include the `.md` extension.` |
| replace_links_in_vault | bool | True | If `True`, then replace the links in the vault pointing to `note` to reflect the new name. |

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    vault_note = VaultNote(temp_vault, rel_path='new_file.md')
    vault_note.create()
    assert vault_note.exists()
    assert vault_note.rel_path in VaultNote.cache[str(temp_vault)]['new_file']

    with ExceptionExpected(ex=FileExistsError):
        vault_note.create() 

    vault_note = VaultNote(temp_vault, rel_path='none_existent_folder/new_file.md')
    with ExceptionExpected(ex=FileNotFoundError):
        vault_note.create()

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    vault_note = VaultNote(temp_vault, name='exponential_function')
    vault_note.delete()
    assert not vault_note.exists()
    assert vault_note.rel_path not in VaultNote.cache[str(temp_vault)]['exponential_function']

If a `VaultNote` object represents an existing file, then the file can be renamed or moved.

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)

    VaultNote.clear_cache()
    vault_note = VaultNote(temp_vault, name='exponential_function')
    vault_note.move_to_folder('')
    assert vault_note.rel_path == 'exponential_function.md'
    assert 'exponential_function.md' in VaultNote.cache[str(temp_vault)]['exponential_function']

The `VaultNote.rename` function renames the file underlying an existing vault note.

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    vault_note = VaultNote(temp_vault, name='exponential_function')
    print(f"Old path: {vault_note.rel_path}")
    vault_note.rename('exp_func')

    expected_path = os.path.normpath('analysis/exp_func.md')
    actual_path = os.path.normpath(vault_note.rel_path)
    test_eq(actual_path, expected_path)

    # test_eq(Path(vault_note.rel_path), Path(r'analysis\exp_func.md'))
    assert vault_note.exists()
    print(f"New path: {vault_note.rel_path}")

Old path: analysis\exponential_function.md
New path: analysis\exp_func.md


By default, the `rename` method replaces links in notes in the vault 

In [ ]:

with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    vault_note = VaultNote(temp_vault, name='note_1')
    other_note = VaultNote(temp_vault, name='note_2_with_links_to_note_1')
    print(f"Old path: {vault_note.rel_path}")
    print(f"`note_2_with_links_to_note_1` is a note with links to `note_1`. The following is its contents:")
    print(other_note.text())
    vault_note.rename('renamed_note_1')

    assert other_note.exists()
    print(f"\nNew path: {vault_note.rel_path}")
    print(f"After `note_1` is renamed, the following is the contents of `note_2_with_links_to_note_1`:")

    new_text_in_note_2 = other_note.text()
    print(new_text_in_note_2)
    assert '[[renamed_note_1]]' in new_text_in_note_2
    assert '[[renamed_note_1|hi]]' in new_text_in_note_2
    assert '[[renamed_note_1#blahblah|hi]]' in new_text_in_note_2
    assert '![[renamed_note_1|embedded]]' in new_text_in_note_2
    assert '[asdf](renamed_note_1)' in new_text_in_note_2
    assert '[asdf](renamed_note_1.md)' in new_text_in_note_2

Old path: topology\note_1.md
`note_2_with_links_to_note_1` is a note with links to `note_1`. The following is its contents:
[[note_1]]

[[note_1|hi]]
[[note_1#blahblah|hi]]

![[note_1|embedded]]

[asdf](note_1)

[asdf](note_1.md)

New path: topology\renamed_note_1.md
After `note_1` is renamed, the following is the contents of `note_2_with_links_to_note_1`:
[[renamed_note_1]]

[[renamed_note_1|hi]]
[[renamed_note_1#blahblah|hi]]

![[renamed_note_1|embedded]]

[asdf](renamed_note_1)

[asdf](renamed_note_1.md)


### Getting a unique note name

It is troublesome to create a note with a non-unique name. The `unique_name` method of the VaultNote class takes a tentative name for a note to be created and adds a number to the name so that the note name will be unique in the vault. 

In [ ]:
#| export obsidian.vault

@patch(cls_method=True)
def unique_name(
        cls: VaultNote,
        name: str, # The base name for the note.
        vault: PathLike, # The vault
        other_unavailable_names: Optional[Sequence[str]] = None # Other names that should be excluded when generating the unique name
        ) -> str: # A str obtained by appending `_{some number}` to the end of `name`.
    r"""A class method to return a name for a note that is unique in
    the vault based on a specified name.
    """
    if not str(vault) in VaultNote.cache:
        cls.update_cache(vault)
    unavailable_names = set(VaultNote.cache[str(vault)])
    if other_unavailable_names is None:
        other_unavailable_names = []
    unavailable_names.update(other_unavailable_names)
    if not name in unavailable_names:
    # if not name in VaultNote.cache[str(vault)]:
        return name
    i = 1
    # while f'{name}_{i}' in VaultNote.cache[str(vault)]:
    while f'{name}_{i}' in unavailable_names:
        i += 1
    return f'{name}_{i}'

In [ ]:
show_doc(VaultNote.unique_name)

---

[source](https://github.com/hyunjongkimmath/trouver/blob/master/trouver/obsidian/vault.py#L719){target="_blank" style="float:right; font-size:smaller"}

### VaultNote.unique_name

>      VaultNote.unique_name (name:str, vault:os.PathLike,
>                             other_unavailable_names:Optional[Sequence[str]]=No
>                             ne)

*A class method to return a name for a note that is unique in
the vault based on a specified name.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| name | str |  | The base name for the note. |
| vault | PathLike |  | The vault |
| other_unavailable_names | Optional | None | Other names that should be excluded when generating the unique name |
| **Returns** | **str** |  | **A str obtained by appending `_{some number}` to the end of `name`.** |

In [ ]:
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    # There is a note named category in the test vault.
    sample_name = VaultNote.unique_name('category', temp_vault)  
    assert not VaultNote(temp_vault, name=sample_name).exists()

    sample_name = VaultNote.unique_name('non_existent_note_name', temp_vault)
    assert not VaultNote(temp_vault, name=sample_name).exists()
    assert sample_name == 'non_existent_note_name'

    sample_name = VaultNote.unique_name('category', temp_vault, other_unavailable_names=['category_1']) 
    assert not VaultNote(temp_vault, name=sample_name).exists()
    assert sample_name not in ['category', 'category_1']

In [ ]:
#| hide
# This import 
with tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir:
    temp_vault = Path(temp_dir) / 'test_vault_1'
    shutil.copytree(_test_directory() / 'test_vault_1', temp_vault)
    # Test _add_single_entry_to_cache
    VaultNote.clear_cache()
    VaultNote._add_single_entry_to_cache(temp_vault, rel_path='directory/file.md')
    assert len(VaultNote.cache) == 1
    assert len(VaultNote.cache[str(temp_vault)]) == 1
    assert len(VaultNote.cache[str(temp_vault)]['file']) == 1

    VaultNote._remove_single_entry_from_cache(temp_vault, rel_path=Path('directory/file.md'))
    assert len(VaultNote.cache) == 1
    assert len(VaultNote.cache[str(temp_vault)]) == 1
    assert len(VaultNote.cache[str(temp_vault)]['file']) == 0

    # 1. Patch the utility function where it is actually used by VaultNote
    with mock.patch('trouver.obsidian.vault.all_paths_to_notes_in_vault') as mock_method:
        VaultNote.update_cache(temp_vault)
        mock_method.assert_called()

    # 2. Patch the class methods on the imported class directly
    # It is often safer/easier to use mock.patch.object for imported classes
    with (mock.patch.object(VaultNote, '_add_single_entry_to_cache') as mock_add_cache,
        mock.patch.object(VaultNote, '_remove_single_entry_from_cache') as mock_remove_cache):
        
        vault_note = VaultNote(temp_vault, name='exponential_function')
        vault_note.move_to_folder('')
        
        mock_remove_cache.assert_called_with(temp_vault, Path('analysis') / Path('exponential_function.md'))
        mock_add_cache.assert_called_with(temp_vault, Path('exponential_function.md'))

In [ ]:
## Copying files in an `Obsidian.md` vault to and from a subvault

In [ ]:
# # TODO: use these methods during vault construction for a reference
# def copy_vault_file_into_subvault(
#         vault: PathLike, # The Path to the vault from which to copy the files.
#         subvaults: Union[PathLike, list[PathLike]], # The Paths to the subvaults to which to copy the files.
#         files: Union[PathLike, list[PathLike]], # The Path to the files, relative to `vault` to copy.
#         replace: bool = True, # If `True`, replace existing files in `subvaults` if necessary. Defaults to `True`
#         backup: bool = True, # If `True` and if `replace=True`, create a backup for any replaced files in a subvault in a folder named `.back` in the root directory of the subvault.
#         ) -> None: 
#     """Copy the specified files in `vault` into subvaults.

#     The files are copied within the subvaults to the same relative paths as
#     they are found in `vault`.

#     Here, "files" include directories. If a directory is copied, then all
#     files and subdirectories of that directory are also copied.

#     **Parameters**
#     - vault - PathLike
#         - The path to the Obsidian vault from which to copy files from.
#     - subvaults - PathLike or list[PathLike]
#         - The paths to the subvaults to which to copy the files.
#     - files - PathLike or list[PathLike]
#         - The files to copy.

#     **Raises**
#     - FileExistsError
#         - If `replace` is `False` and some subvault already has a file at
#           the path in which a file-copy is attempted. In this case, no
#           files are copied.
#     - FileNotFoundError
#         - If a path specified in `files` does not exist in `vault`. In this
#           case, no files are copied.

#     """
#     vault = Path(vault)
#     if isinstance(subvaults, PathLike):
#         subvaults = [subvaults]
#     if isinstance(files, PathLike):

#         files = [files]


#     # TODO: Implement this as a private function 
#     for file in files:
#         if not os.path.exists(vault / file):
#             raise FileNotFoundError(
#                 f"Attempted to copy files/folders from a vault into subvaults"
#                 f", but there is at least one non-existent files"
#                 f". No files have been copied."
#                 f"vault: {vault}"
#                 f"file: {file}")

#     if not replace: 
#         for subvault, file in itertools.product(subvaults, files):
#             if os.path.exists(subvault / file):
#                 raise FileExistsError(
#                     f"Attempted to copy files/folders from a vault into subvaults"
#                     f", but at least one subvault already has a file that is"
#                     f" supposed to be copied from the vault"
#                     f". No files have been copied."
#                     f"subvault: {subvault}"
#                     f"file: {file}")
    
#     # TODO: copy and backup files

#     return

In [ ]:
#| export obsidian.vault
def test_function():
    return None